importing the dependencies


In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
data = pd.read_csv(r'Datasets/train.csv')

In [3]:
data

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0
...,...,...,...,...,...,...,...,...,...
57472,4294656694,gpt-4-0613,claude-1,"[""A simple mnemonic for \u03c0:\n\""How I wish ...","[""Sure, let's break it down:\n\n1. \""How\"" has...","[""Here is how that mnemonic represents the dig...",1,0,0
57473,4294692063,claude-2.0,llama-2-13b-chat,"[""In python, implement a naive Bayes with gaus...","[""Here is an implementation of a naive Bayes c...","[""Sure! Here's an implementation of a naive Ba...",1,0,0
57474,4294710549,claude-1,alpaca-13b,"[""is it unethical to work on building weapons?...","[""Working on weapons technology raises some et...","[""It depends on the context. Weapons can be us...",1,0,0
57475,4294899228,palm-2,tulu-2-dpo-70b,"[""If a bait contains 0,0025% bromadiolon then ...","[""Bromadiolone is a rodenticide which is most ...","[""As an AI language model, I do not promote or...",0,1,0


In [4]:
data.columns

Index(['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b',
       'winner_model_a', 'winner_model_b', 'winner_tie'],
      dtype='object')

In [5]:
Test_data = pd.read_csv(r'Datasets/test.csv')

In [6]:
Test_data.head()

,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran..."
1,211333,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ..."
2,1233961,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p..."


In [7]:
Test_data = pd.concat([Test_data, data[['model_a', 'model_b']]], axis=1)


In [8]:
Test_data = Test_data.head(3)


In [9]:
Test_data

,id,prompt,response_a,response_b,model_a,model_b
0,136060.0,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran...",gpt-4-1106-preview,gpt-4-0613
1,211333.0,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ...",koala-13b,gpt-4-0613
2,1233961.0,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p...",gpt-3.5-turbo-0613,mistral-medium


In [10]:
Test_data.isnull().sum(
    
)

id            0
prompt        0
response_a    0
response_b    0
model_a       0
model_b       0
dtype: int64

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              57477 non-null  int64 
 1   model_a         57477 non-null  object
 2   model_b         57477 non-null  object
 3   prompt          57477 non-null  object
 4   response_a      57477 non-null  object
 5   response_b      57477 non-null  object
 6   winner_model_a  57477 non-null  int64 
 7   winner_model_b  57477 non-null  int64 
 8   winner_tie      57477 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 3.9+ MB


In [12]:
data['model_b'].nunique()

64

In [13]:
data['model_a']

0        gpt-4-1106-preview
1                 koala-13b
2        gpt-3.5-turbo-0613
3          llama-2-13b-chat
4                 koala-13b
                ...        
57472            gpt-4-0613
57473            claude-2.0
57474              claude-1
57475                palm-2
57476    gemini-pro-dev-api
Name: model_a, Length: 57477, dtype: object

In [14]:
data.drop(columns='id',inplace=True)
Test_data.drop(columns='id',inplace=True)

In [15]:
label = LabelEncoder()

data['model_a'] = label.fit_transform(data['model_a'])
data['model_b'] = label.fit_transform(data['model_b'])
Test_data['model_a'] = label.fit_transform(Test_data['model_a'])
Test_data['model_b'] = label.fit_transform(Test_data['model_b'])

In [16]:
data

,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,24,23,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,27,23,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,19,35,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,29,33,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,27,18,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0
...,...,...,...,...,...,...,...,...
57472,23,5,"[""A simple mnemonic for \u03c0:\n\""How I wish ...","[""Sure, let's break it down:\n\n1. \""How\"" has...","[""Here is how that mnemonic represents the dig...",1,0,0
57473,6,29,"[""In python, implement a naive Bayes with gaus...","[""Here is an implementation of a naive Bayes c...","[""Sure! Here's an implementation of a naive Ba...",1,0,0
57474,5,1,"[""is it unethical to work on building weapons?...","[""Working on weapons technology raises some et...","[""It depends on the context. Weapons can be us...",1,0,0
57475,44,55,"[""If a bait contains 0,0025% bromadiolon then ...","[""Bromadiolone is a rodenticide which is most ...","[""As an AI language model, I do not promote or...",0,1,0


In [17]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Musta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))


In [19]:
# Function to stem and clean the text
def stem_and_clean(text):
    words = text.split()
    words = [ps.stem(word) for word in words if word.lower() not in stop_words]
    return ' '.join(words)

In [20]:
data['prompt'] = data['prompt'].apply(stem_and_clean)
data['response_a'] = data['response_a'].apply(stem_and_clean)
data['response_b'] = data['response_b'].apply(stem_and_clean)
Test_data['prompt'] = Test_data['prompt'].apply(stem_and_clean)
Test_data['response_a'] = Test_data['response_a'].apply(stem_and_clean)
Test_data['response_b'] = Test_data['response_b'].apply(stem_and_clean)


In [21]:
Test_data.head()

,prompt,response_a,response_b,model_a,model_b
0,"[""i three orang today, ate orang yesterday. ma...","[""you two orang today.""]","[""you still three oranges. eat orang yesterday...",1,0
1,"[""you mediat heat polit debat two oppos partie...","[""thank share detail situation. mediator, unde...","[""mr reddi ms blue valid point arguments. one ...",2,0
2,"[""how initi classif head transfer learning. ex...","[""when want initi classif head transfer learni...","[""to initi classif head perform transfer learn...",0,1


In [22]:
from sklearn.feature_extraction.text import HashingVectorizer
import numpy as np

# Initialize HashingVectorizer (fixed-size vector space)
vectorizer = HashingVectorizer(n_features=10000)  # You can adjust the number of features

# Define a function to vectorize each column and store as a dense array format
def data_vectorize_column(data, columns):
    for col in columns:
        # Ensure the column contains strings, converting them if necessary
        data[col] = data[col].astype(str)
        
        # Fit and transform the column to create a sparse matrix
        X = vectorizer.transform(data[col])
        
        # Convert the sparse matrix row-by-row to dense format
        data[col] = [X.getrow(i).toarray().flatten() for i in range(X.shape[0])]
        
    return data

def Testdata_vectorize_column(Test_data, columns):
    for col in columns:
        # Ensure the column contains strings, converting them if necessary
        Test_data[col] = Test_data[col].astype(str)
        
        # Fit and transform the column to create a sparse matrix
        X = vectorizer.transform(Test_data[col])
        
        # Convert the sparse matrix row-by-row to dense format
        Test_data[col] = [X.getrow(i).toarray().flatten() for i in range(X.shape[0])]
    
    return Test_data

# List of columns to vectorize
columns_to_vectorize = ['prompt', 'response_a', 'response_b']

# Apply the vectorization to both data and Test_data DataFrames
data = data_vectorize_column(data, columns_to_vectorize)
Test_data = Testdata_vectorize_column(Test_data, columns_to_vectorize)

# Display the updated DataFrames
print("Vectorized Data:")
print(data)

print("\nVectorized Test Data:")
print(Test_data)


Vectorized Data:
       model_a  model_b                                             prompt  \
0           24       23  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1           27       23  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2           19       35  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3           29       33  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4           27       18  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
...        ...      ...                                                ...   
57472       23        5  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
57473        6       29  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
57474        5        1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
57475       44       55  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
57476       16       24  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                                              

In [23]:
data.shape

(57477, 8)

In [24]:
data.head(1)

,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,24,23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0330049180992...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0,0


In [25]:
"""import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Assume you have your X_train as a DataFrame with 'prompt', 'model_a', 'model_b'
dara = data[['prompt', 'model_a', 'model_b']]  # X_train will be all columns in this case

# If 'prompt' is a list, process it accordingly (flatten, apply PCA, etc.)
X['prompt'] = X['prompt'].apply(lambda x: np.fromstring(x.strip('[]'), sep=', '))

# Example: Normalize 'model_a' and 'model_b'
scaler = StandardScaler()
data[['model_a', 'model_b']] = scaler.fit_transform(X[['model_a', 'model_b']])
"""

"import numpy as np\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\n\n# Assume you have your X_train as a DataFrame with 'prompt', 'model_a', 'model_b'\ndara = data[['prompt', 'model_a', 'model_b']]  # X_train will be all columns in this case\n\n# If 'prompt' is a list, process it accordingly (flatten, apply PCA, etc.)\nX['prompt'] = X['prompt'].apply(lambda x: np.fromstring(x.strip('[]'), sep=', '))\n\n# Example: Normalize 'model_a' and 'model_b'\nscaler = StandardScaler()\ndata[['model_a', 'model_b']] = scaler.fit_transform(X[['model_a', 'model_b']])\n"

Splitting the data in X and Y

In [26]:
X = data.drop(columns=['winner_model_a', 'winner_model_b', 'winner_tie'])
Y = data[['winner_model_a', 'winner_model_b', 'winner_tie']]
"""Y_response_a = data[['winner_model_a', 'winner_model_b', 'winner_tie']]
Y_response_ = data[['winner_model_a', 'winner_model_b', 'winner_tie']]"""


"Y_response_a = data[['winner_model_a', 'winner_model_b', 'winner_tie']]\nY_response_ = data[['winner_model_a', 'winner_model_b', 'winner_tie']]"

In [27]:
Y.shape

(57477, 3)

In [28]:
X

,model_a,model_b,prompt,response_a,response_b
0,24,23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0330049180992...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,27,23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,19,35,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,29,33,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,27,18,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...
57472,23,5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57473,6,29,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57474,5,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, -0.014628577924855639, 0.0, 0.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
57475,44,55,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   model_a         57477 non-null  int32 
 1   model_b         57477 non-null  int32 
 2   prompt          57477 non-null  object
 3   response_a      57477 non-null  object
 4   response_b      57477 non-null  object
 5   winner_model_a  57477 non-null  int64 
 6   winner_model_b  57477 non-null  int64 
 7   winner_tie      57477 non-null  int64 
dtypes: int32(2), int64(3), object(3)
memory usage: 3.1+ MB


In [30]:
"""import numpy as np

# Flatten the arrays by concatenating them
Y = np.hstack([Y['winner_model_a'].apply(lambda x: np.array(x).flatten()),
               Y['winner_model_b'].apply(lambda x: np.array(x).flatten()),
               Y['winner_tie'].apply(lambda x: np.array(x).flatten())])"""


"import numpy as np\n\n# Flatten the arrays by concatenating them\nY = np.hstack([Y['winner_model_a'].apply(lambda x: np.array(x).flatten()),\n               Y['winner_model_b'].apply(lambda x: np.array(x).flatten()),\n               Y['winner_tie'].apply(lambda x: np.array(x).flatten())])"

In [31]:
"""X = np.hstack([X['prompt'].apply(lambda x: np.array(x).flatten()),
               X['response_a'].apply(lambda x: np.array(x).flatten()),
               X['response_b'].apply(lambda x: np.array(x).flatten())])"""


"X = np.hstack([X['prompt'].apply(lambda x: np.array(x).flatten()),\n               X['response_a'].apply(lambda x: np.array(x).flatten()),\n               X['response_b'].apply(lambda x: np.array(x).flatten())])"

Splitting with train test split

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [33]:
Y_train

,winner_model_a,winner_model_b,winner_tie
30205,0,0,1
8763,0,1,0
30501,0,1,0
33619,0,0,1
22573,0,1,0
...,...,...,...
54343,0,0,1
38158,0,0,1
860,0,1,0
15795,0,1,0


In [34]:
print(X.shape,X_train.shape,X_test.shape)

(57477, 5) (45981, 5) (11496, 5)


Model Implementation

In [35]:
model = MultinomialNB()


In [36]:
X_train

,model_a,model_b,prompt,response_a,response_b
30205,61,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
8763,6,23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
30501,52,40,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
33619,7,24,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
22573,21,23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, -0.0602475233128778, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...
54343,63,33,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
38158,60,8,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
860,2,8,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
15795,0,26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.11805626722019105, 0.0, 0.0, 0.0,..."


In [47]:
Y_train

,winner_model_a,winner_model_b,winner_tie
30205,0,0,1
8763,0,1,0
30501,0,1,0
33619,0,0,1
22573,0,1,0
...,...,...,...
54343,0,0,1
38158,0,0,1
860,0,1,0
15795,0,1,0


In [37]:
print(Y_train.ndim)  # Output: 1 for 1D, 2 for 2D


2


In [ ]:
X_train.ndim

2

In [39]:
#model.fit(X_train,Y_train)

In [40]:
"""from sklearn.naive_bayes import MultinomialNB
import pandas as pd

# Initialize the models
a_model = MultinomialNB()
b_model = MultinomialNB()
c_model = MultinomialNB()

# Ensure that the target columns are 1D arrays (flatten them if necessary)
# Use .values.ravel() to make sure the target is a 1D array (a Series or flat array)
a_model.fit(X_train, Y_train['winner_model_a'].values.ravel())
b_model.fit(X_train, Y_train['winner_model_b'].values.ravel())
c_model.fit(X_train, Y_train['winner_tie'].values.ravel())

# Make predictions on the test data
a_predictions = a_model.predict(X_test)
b_predictions = b_model.predict(X_test)
c_predictions = c_model.predict(X_test)

# Combine predictions into a final DataFrame
final_predictions = pd.DataFrame({
    'winner_model_a': a_predictions,
    'winner_model_b': b_predictions,
    'winner_tie': c_predictions
})

# Display the final predictions
print(final_predictions)
"""

"from sklearn.naive_bayes import MultinomialNB\nimport pandas as pd\n\n# Initialize the models\na_model = MultinomialNB()\nb_model = MultinomialNB()\nc_model = MultinomialNB()\n\n# Ensure that the target columns are 1D arrays (flatten them if necessary)\n# Use .values.ravel() to make sure the target is a 1D array (a Series or flat array)\na_model.fit(X_train, Y_train['winner_model_a'].values.ravel())\nb_model.fit(X_train, Y_train['winner_model_b'].values.ravel())\nc_model.fit(X_train, Y_train['winner_tie'].values.ravel())\n\n# Make predictions on the test data\na_predictions = a_model.predict(X_test)\nb_predictions = b_model.predict(X_test)\nc_predictions = c_model.predict(X_test)\n\n# Combine predictions into a final DataFrame\nfinal_predictions = pd.DataFrame({\n    'winner_model_a': a_predictions,\n    'winner_model_b': b_predictions,\n    'winner_tie': c_predictions\n})\n\n# Display the final predictions\nprint(final_predictions)\n"

In [41]:
"""from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Split the target Y into separate columns
Y_train_model_a = Y_train['winner_model_a']
Y_train_model_b = Y_train['winner_model_b']
Y_train_winner_tie = Y_train['winner_tie']

Y_test_model_a = Y_test['winner_model_a']
Y_test_model_b = Y_test['winner_model_b']
Y_test_winner_tie = Y_test['winner_tie']

# Create a function to fit, predict, and store predictions for each column
def train_and_predict(X_train, Y_train, X_test, model):
    model.fit(X_train, Y_train)
    predictions = model.predict(X_test)
    return predictions

# Initialize the base model (Naive Bayes in this case)
base_model = GaussianNB()

# Create separate models for each target and predict
model_a_predictions = train_and_predict(X_train, Y_train_model_a, X_test, base_model)
model_b_predictions = train_and_predict(X_train, Y_train_model_b, X_test, base_model)
winner_tie_predictions = train_and_predict(X_train, Y_train_winner_tie, X_test, base_model)

# Combine the predictions into a final dataframe
final_predictions = pd.DataFrame({
    'winner_model_a': model_a_predictions,
    'winner_model_b': model_b_predictions,
    'winner_tie': winner_tie_predictions
})

# Display the final prediction dataframe
print("Final Predictions:")
print(final_predictions)
"""

'from sklearn.model_selection import train_test_split\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import LabelEncoder\nimport pandas as pd\n\n# Split the target Y into separate columns\nY_train_model_a = Y_train[\'winner_model_a\']\nY_train_model_b = Y_train[\'winner_model_b\']\nY_train_winner_tie = Y_train[\'winner_tie\']\n\nY_test_model_a = Y_test[\'winner_model_a\']\nY_test_model_b = Y_test[\'winner_model_b\']\nY_test_winner_tie = Y_test[\'winner_tie\']\n\n# Create a function to fit, predict, and store predictions for each column\ndef train_and_predict(X_train, Y_train, X_test, model):\n    model.fit(X_train, Y_train)\n    predictions = model.predict(X_test)\n    return predictions\n\n# Initialize the base model (Naive Bayes in this case)\nbase_model = GaussianNB()\n\n# Create separate models for each target and predict\nmodel_a_predictions = train_and_predict(X_train, Y_train_model_a, X_test, base_model)\nmodel_b_pr

In [53]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Assuming you've loaded your dataset into X and Y already

# Flatten 'response_a' and 'response_b' into individual features
# Each element of the arrays becomes a separate feature
response_a_features = pd.DataFrame(X['response_a'].tolist(), columns=[f'response_a_{i}' for i in range(len(X['response_a'][0]))])
response_b_features = pd.DataFrame(X['response_b'].tolist(), columns=[f'response_b_{i}' for i in range(len(X['response_b'][0]))])

# Concatenate the original features (model_a, model_b, prompt) with the flattened arrays
X_transformed = pd.concat([X[['model_a', 'model_b', 'prompt']], response_a_features, response_b_features], axis=1)

# Now we have a properly shaped dataset where each row is a sample and columns are individual features

# Split the dataset into training and test sets (for demonstration purposes, you may already have separate test data)
X_train, X_test, Y_train, Y_test = train_test_split(X_transformed, Y, test_size=0.2, random_state=42)

# Train a RandomForestClassifier (you can choose other classifiers if you wish)
model = RandomForestClassifier()
model.fit(X_train, Y_train)

# Predict probabilities for each class (winner_model_a, winner_model_b, winner_tie)
probabilities = model.predict_proba(X_test)

# Create the submission DataFrame
submission = pd.DataFrame({
    'id': [136060, 211333, 1233961],  # Example IDs for submission
    'winner_model_a': probabilities[:, 0],  # Probabilities for class 0 (model_a)
    'winner_model_b': probabilities[:, 1],  # Probabilities for class 1 (model_b)
    'winner_tie': probabilities[:, 2]  # Probabilities for class 2 (tie)
})

# Save the submission to a CSV file
submission.to_csv('submission.csv', index=False)

print("Submission file has been saved!")


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices